# Training English Sentiment Model

IMDb movie reviews · binary **Positive / Negative** (no Neutral).

## 1. Imports

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

ROOT = Path("..") if (Path("..") / "data").exists() else Path(".")
sys.path.insert(0, str(ROOT.resolve()))

from src.English_model import (
    ENGLISH_FEATURE_NAMES,
    ENGLISH_MODEL_NAMES,
    EnglishSentimentModel,
    build_english_baseline,
    build_english_pipeline,
    make_classifier,
    make_vectorizer,
    save_english_model,
)
from src.labels import ENGLISH, NEGATIVE, POSITIVE, map_sentiment
from src.Preprocessing_pipeline import preprocess_english

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:.4f}".format)
LABELS = [NEGATIVE, POSITIVE]


## 2. Load Data

In [2]:
DATA = ROOT / "data" / "raw" / "english" / "MovieReviewTrainingDatabase.csv"
df = pd.read_csv(DATA)
print(DATA.resolve())
df.head()


/home/abdlrhman/Courses/IEEE SSCS/NLP/Week1/Project1/data/raw/english/MovieReviewTrainingDatabase.csv


,sentiment,review
0,Positive,"With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documenta..."
1,Positive,'The Classic War of the Worlds' by Timothy Hines is a very entertaining film that obviously goes to great effort and...
2,Negative,The film starts with a manager (Nicholas Bell) giving welcome investors (Robert Carradine) to Primal Park . A secret...
3,Negative,"It must be assumed that those who praised this film ('the greatest filmed opera ever,' didn't I read somewhere?) eit..."
4,Positive,"Superbly trashy and wondrously unpretentious 80's exploitation, hooray! The pre-credits opening sequences somewhat g..."


## 3. EDA

In [3]:
print("shape:", df.shape)
print("columns:", list(df.columns))
print(df["sentiment"].value_counts())
print(df.isnull().sum())
print("duplicate reviews:", int(df.duplicated(subset=["review"]).sum()))

fig, ax = plt.subplots(figsize=(5, 3))
df["sentiment"].value_counts().reindex(["Positive", "Negative"]).plot(
    kind="bar", ax=ax, color=["#2ca02c", "#d62728"], rot=0
)
ax.set_title("English class counts")
plt.tight_layout()
plt.show()


shape: (25000, 2)
columns: ['sentiment', 'review']
sentiment
Positive    12500
Negative    12500
Name: count, dtype: int64
sentiment    0
review       0
dtype: int64
duplicate reviews: 96


/tmp/ipykernel_56758/343983022.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Cleaning

In [4]:
n0 = len(df)
df = df.drop_duplicates(subset=["review"]).copy()
df["label"] = df["sentiment"].map(lambda x: map_sentiment(x, ENGLISH))
print(f"dropped {n0 - len(df)} duplicate reviews → {len(df)}")
print(df["label"].value_counts())
assert set(df["label"]) == {POSITIVE, NEGATIVE}


dropped 96 duplicate reviews → 24904
label
Positive    12472
Negative    12432
Name: count, dtype: int64


## 5. Preprocessing

English only: lowercase → URLs → HTML → unwanted characters → whitespace. No Arabic steps.

In [5]:
df["clean_text"] = df["review"].map(preprocess_english)
print(df[["review", "clean_text", "label"]].head(2).assign(
    review=lambda x: x["review"].str.slice(0, 90),
    clean_text=lambda x: x["clean_text"].str.slice(0, 90),
))


                                                                                       review  \
0  With all this stuff going down at the moment with MJ i've started listening to his music,    
1  'The Classic War of the Worlds' by Timothy Hines is a very entertaining film that obviousl   

                                                                                   clean_text  \
0  with all this stuff going down at the moment with mj i've started listening to his music w   
1  'the classic war of the worlds' by timothy hines is a very entertaining film that obviousl   

      label  
0  Positive  
1  Positive  


## 6. Train/Test Split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)
print("train", len(X_train), "val", len(X_val), "test", len(X_test))


train 19923 val 3985 test 4981


## 7. Feature Extraction

Classical TF-IDF / Bag of Words. `TfidfVectorizer(ngram_range=(1, 2), max_features=30000)` is the baseline.

## 8. Baseline Model

TF-IDF unigram+bigram + Logistic Regression — a working pipeline, not a tuned one.

In [7]:
def scores(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, pos_label=POSITIVE, zero_division=0),
        "Recall": recall_score(y_true, y_pred, pos_label=POSITIVE, zero_division=0),
        "F1": f1_score(y_true, y_pred, pos_label=POSITIVE, zero_division=0),
    }

baseline = build_english_baseline()
baseline.fit(X_train, y_train)
base_pred = baseline.predict(X_test)
print(classification_report(y_test, base_pred, labels=LABELS, digits=4))
print(pd.Series(scores(y_test, base_pred), name="baseline"))


              precision    recall  f1-score   support

    Negative     0.9046    0.8846    0.8944      2486
    Positive     0.8875    0.9070    0.8971      2495

    accuracy                         0.8958      4981
   macro avg     0.8960    0.8958    0.8958      4981
weighted avg     0.8960    0.8958    0.8958      4981

Accuracy    0.8958
Precision   0.8875
Recall      0.9070
F1          0.8971
Name: baseline, dtype: float64


## 9. Model Experiments

Naive Bayes / Logistic Regression / Linear SVM × TF-IDF uni / uni+bi / Bag of Words. Selection is **validation F1**, test stays held out.

In [8]:
rows = []
for feature_name in ENGLISH_FEATURE_NAMES:
    vec = make_vectorizer(feature_name)
    Xtr = vec.fit_transform(X_fit)
    Xva = vec.transform(X_val)
    for model_name in ENGLISH_MODEL_NAMES:
        clf = make_classifier(model_name)
        clf.fit(Xtr, y_fit)
        pred = clf.predict(Xva)
        row = {"Model": model_name, "Features": feature_name, **scores(y_val, pred)}
        rows.append(row)
        print(f"val {model_name:20} {feature_name:24} F1={row['F1']:.4f}")

val_results = pd.DataFrame(rows).sort_values("F1", ascending=False).reset_index(drop=True)
val_results


val Naive Bayes          TF-IDF unigram           F1=0.8545


val Logistic Regression  TF-IDF unigram           F1=0.8822


val Linear SVM           TF-IDF unigram           F1=0.8849


val Naive Bayes          TF-IDF unigram + bigram  F1=0.8772


val Logistic Regression  TF-IDF unigram + bigram  F1=0.8862


...

val Naive Bayes          Bag of Words             F1=0.8374


val Logistic Regression  Bag of Words             F1=0.8760


val Linear SVM           Bag of Words             F1=0.8584


/home/abdlrhman/miniconda3/lib/python3.13/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


,Model,Features,Accuracy,Precision,Recall,F1
0,Linear SVM,TF-IDF unigram + bigram,0.8901,0.8864,0.8953,0.8908
1,Logistic Regression,TF-IDF unigram + bigram,0.8848,0.8769,0.8958,0.8862
2,Linear SVM,TF-IDF unigram,0.8836,0.8766,0.8933,0.8849
3,Logistic Regression,TF-IDF unigram,0.8806,0.8715,0.8933,0.8822
4,Naive Bayes,TF-IDF unigram + bigram,0.8770,0.8776,0.8768,0.8772
5,Logistic Regression,Bag of Words,0.8748,0.8689,0.8833,0.8760
6,Linear SVM,Bag of Words,0.8577,0.8556,0.8612,0.8584
7,Naive Bayes,TF-IDF unigram,0.8570,0.8710,0.8387,0.8545
8,Naive Bayes,Bag of Words,0.8417,0.8621,0.8141,0.8374


## 10. Evaluation

In [9]:
best_row = val_results.iloc[0]
best_model_name, best_feature_name = best_row["Model"], best_row["Features"]
print("validation winner:", best_model_name, "+", best_feature_name)

best = build_english_pipeline(best_feature_name, best_model_name)
best.fit(X_train, y_train)
y_pred = best.predict(X_test)
print("\nHeld-out TEST")
print(classification_report(y_test, y_pred, labels=LABELS, digits=4))
print(pd.Series(scores(y_test, y_pred), name="test"))

cm = confusion_matrix(y_test, y_pred, labels=LABELS)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(cmap="Blues", ax=ax, colorbar=False)
ax.set_title("English — test confusion matrix")
plt.tight_layout()
plt.show()


validation winner: Linear SVM + TF-IDF unigram + bigram



Held-out TEST
              precision    recall  f1-score   support

    Negative     0.8964    0.8837    0.8900      2486
    Positive     0.8858    0.8982    0.8919      2495

    accuracy                         0.8910      4981
   macro avg     0.8911    0.8910    0.8910      4981
weighted avg     0.8911    0.8910    0.8910      4981

Accuracy    0.8910
Precision   0.8858
Recall      0.8982
F1          0.8919
Name: test, dtype: float64


/tmp/ipykernel_56758/4194844423.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Error analysis

Classical bag-of-n-grams misses **negation scope** and **sarcasm**.

| Example | Likely gold | Typical failure |
|---|---|---|
| The movie wasn't bad. | Positive | Negation of *bad* |
| Great, another terrible movie. | Negative | Sarcasm / mixed cues |

In [10]:
from src.English_model import EnglishSentimentModel

tmp = EnglishSentimentModel(best)
for text in (
    "The movie wasn't bad.",
    "Great, another terrible movie.",
    "This movie is not good.",
):
    print(f"{tmp.predict(text):10}  {text}")


Negative    The movie wasn't bad.
Negative    Great, another terrible movie.
Negative    This movie is not good.


## 11. Best Model

In [11]:
print(best)
print(best_row.to_dict())


Pipeline(steps=[('vec',
                 TfidfVectorizer(max_features=30000, ngram_range=(1, 2))),
                ('clf', LinearSVC())])
{'Model': 'Linear SVM', 'Features': 'TF-IDF unigram + bigram', 'Accuracy': 0.8900878293601003, 'Precision': 0.8864087301587301, 'Recall': 0.8952905811623246, 'F1': 0.890827517447657}


## 12. Save Model

Entire sklearn Pipeline (vectorizer + classifier), not the classifier alone.

In [12]:
path = save_english_model(best, ROOT / "models" / "English_model_weights.pkl")
print("saved", path.resolve())


saved /home/abdlrhman/Courses/IEEE SSCS/NLP/Week1/Project1/models/English_model_weights.pkl


## 13. Example Predictions

In [13]:
model = EnglishSentimentModel.load(path)
for text in (
    "I absolutely loved this movie.",
    "This was one of the worst movies I've ever watched.",
    "This movie was amazing.",
    "I hated every minute of it.",
):
    print(f"{model.predict(text):10}  {text}")


Positive    I absolutely loved this movie.
Negative    This was one of the worst movies I've ever watched.
Positive    This movie was amazing.
Negative    I hated every minute of it.
